In [145]:
import numpy as np
import pandas as pd
import re

In [146]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [147]:
df = pd.read_csv('C:\\Users\\ACER\Desktop\\Projects\\Machine Learning Based Real Estate System\\Dataset\\Cleaned_properties_v1.csv')

df.head(1)

,property_type,society,sector,price,price_per_sqft,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,floorNum,facing,agePossession,nearbyLocations,furnishDetails,features
0,flat,umang winter hills,sector 77,1.0,7500.0,1333.0,Carpet area: 1340 (124.49 sq.m.),2,2,2,not available,3.0,NaN,0 to 1 Year Old,"['Entertainland Mall', 'Delhi Jaipur Expressway', 'Jhankar Senior Secondary School', 'Singhania University, Manesar', 'Miracles Apollo Hospital', 'Indira Gandhi International Airport', 'Garhi Harsaru Junction', 'Eros Corporate Park', 'Hyatt Regency Gurgaon', 'Aravalli Hills']",NaN,"['Security / Fire Alarm', 'Feng Shui / Vaastu Compliant', 'Intercom Facility', 'Lift(s)', 'Maintenance Staff', 'Piped-gas', 'Visitor Parking', 'Swimming Pool', 'Park', 'Security Personnel', 'Fitness Centre / GYM', 'Rain Water Harvesting', 'Club house / Community Center']"


In [148]:
df.shape

(3856, 17)

`Focus is on ->`

`areaWithType, additionalRoom, agePossession, furnishDetails, features`

### 1. areaWithType

In [149]:
df.sample(5)[['price', 'area', 'areaWithType']]

,price,area,areaWithType
3076,24.00,400.0,Plot area 400(37.16 sq.m.)
3684,1.60,1929.0,Super Built up area 1929(179.21 sq.m.)Built Up area: 1548 sq.ft. (143.81 sq.m.)Carpet area: 1300 sq.ft. (120.77 sq.m.)
2652,2.15,1433.0,Built Up area: 1433 (133.13 sq.m.)
345,1.40,1447.0,Super Built up area 1720(159.79 sq.m.)Carpet area: 1447 sq.ft. (134.43 sq.m.)
2541,1.72,2200.0,Carpet area: 2200 (204.39 sq.m.)


In [150]:
# This function extracts the Super Built up Area

def get_super_built_up_area(text):
    match = re.search('Super Built up area (\d+\.?\d+)', text)
    if match:
        return float(match.group(1))
    return None

In [151]:
# This function extracts the Built up Area or Carpet Area

def get_area(text, area_type):
    match = re.search(area_type +  r'\s*:\s*(\d+\.?\d*)', text)
    if match:
        return float(match.group(1))
    return None

In [152]:
# This function checks if the area is provided in sq.m and converts it to sqft if needed

def convert_to_sqft(text, area_value):
    if area_value is None:
        return None
    
    match = re.search(r'{} \((\d+\.?\d*) sq.m. \)'.format(area_value), text)
    if match:
        sq_m_value = float(match.group(1))
        return sq_m_value * 10.7639 # Conversion factor from sq.m. to sqft
    return area_value

In [153]:
# Extract Super Built up area and Convert to sqft if Needed
df['super_built_up_area'] = df['areaWithType'].apply(get_super_built_up_area)
df['super_built_up_area'] = df.apply(lambda x: convert_to_sqft(x['areaWithType'], x['super_built_up_area']), axis=1)

# Extract Build Up area and convert to sqft if needed
df['built_up_area'] = df['areaWithType'].apply(lambda x: get_area(x, 'Built Up area'))
df['built_up_area'] = df.apply(lambda x: convert_to_sqft(x['areaWithType'], x['built_up_area']), axis=1)

# Extract Carpet Area and Convert to sqft if needed
df['carpet_area'] = df['areaWithType'].apply(lambda x: get_area(x, 'Carpet area'))
df['carpet_area'] = df.apply(lambda x: convert_to_sqft(x['areaWithType'], x['carpet_area']), axis=1)

In [154]:
df[['price', 'property_type', 'area', 'areaWithType', 'super_built_up_area', 'built_up_area', 'carpet_area']].sample(5)

,price,property_type,area,areaWithType,super_built_up_area,built_up_area,carpet_area
2893,1.25,flat,1447.0,Carpet area: 1447 (134.43 sq.m.),NaN,NaN,1447.0
1461,1.75,flat,3059.0,Super Built up area 2727(253.35 sq.m.),2727.0,NaN,NaN
3451,2.50,flat,1850.0,Super Built up area 1850(171.87 sq.m.),1850.0,NaN,NaN
364,1.15,flat,1350.0,Super Built up area 1350(125.42 sq.m.),1350.0,NaN,NaN
2016,1.75,flat,1950.0,Built Up area: 1950 (181.16 sq.m.),NaN,1950.0,NaN


In [155]:
df[~((df['super_built_up_area'].isnull()) | (df['built_up_area'].isnull()) | (df['carpet_area'].isnull()))][['price', 'property_type', 'area', 'areaWithType', 'super_built_up_area', 'built_up_area', 'carpet_area']].shape

(536, 7)

In [156]:
df[df['areaWithType'].str.contains('Plot')][['price', 'property_type', 'area', 'areaWithType', 'super_built_up_area', 'built_up_area', 'carpet_area']].shape

(710, 7)

In [157]:
df.isnull().sum()

property_type             0
society                   1
sector                    0
price                    18
price_per_sqft           18
area                     18
areaWithType              0
bedRoom                   0
bathroom                  0
balcony                   0
additionalRoom            0
floorNum                 21
facing                 1127
agePossession             1
nearbyLocations         184
furnishDetails         1001
features                661
super_built_up_area    1935
built_up_area          2645
carpet_area            1891
dtype: int64

In [158]:
all_nan_df = df[((df['super_built_up_area'].isnull()) & (df['built_up_area'].isnull()) & (df['carpet_area'].isnull()))][['price', 'property_type', 'area', 'areaWithType', 'super_built_up_area', 'built_up_area', 'carpet_area']]

In [159]:
all_nan_df.head()

,price,property_type,area,areaWithType,super_built_up_area,built_up_area,carpet_area
9,0.75,house,600.0,Plot area 600(55.74 sq.m.),NaN,NaN,NaN
12,5.60,house,3240.0,Plot area 360(301.01 sq.m.),NaN,NaN,NaN
15,0.48,house,80.0,Plot area 80(7.43 sq.m.),NaN,NaN,NaN
19,8.50,house,6300.0,Plot area 6300(585.29 sq.m.),NaN,NaN,NaN
23,0.92,house,603.0,Plot area 67(56.02 sq.m.),NaN,NaN,NaN


In [160]:
all_nan_index = df[((df['super_built_up_area'].isnull()) & (df['built_up_area'].isnull()) & (df['carpet_area'].isnull()))][['price', 'property_type', 'area', 'areaWithType', 'super_built_up_area', 'built_up_area', 'carpet_area']].index

In [161]:
# Function to Extract Plot Area from 'areaWithType' column

def extract_plot_area(area_with_type):
    match = re.search(r'Plot area (\d+\.?\d*)', area_with_type)
    return float(match.group(1)) if match else None

In [162]:
all_nan_df['built_up_area'] = all_nan_df['areaWithType'].apply(extract_plot_area)

In [163]:
all_nan_df.head()

,price,property_type,area,areaWithType,super_built_up_area,built_up_area,carpet_area
9,0.75,house,600.0,Plot area 600(55.74 sq.m.),NaN,600.0,NaN
12,5.60,house,3240.0,Plot area 360(301.01 sq.m.),NaN,360.0,NaN
15,0.48,house,80.0,Plot area 80(7.43 sq.m.),NaN,80.0,NaN
19,8.50,house,6300.0,Plot area 6300(585.29 sq.m.),NaN,6300.0,NaN
23,0.92,house,603.0,Plot area 67(56.02 sq.m.),NaN,67.0,NaN


In [164]:
def convert_scale(row):
    if np.isnan(row['area']) or np.isnan(row['built_up_area']):
        return row['built_up_area']
    else:
        if round(row['area']/row['built_up_area']) == 9.0:
            return row['built_up_area'] * 9
        elif round(row['area']/row['built_up_area']) == 11.0:
            return row['built_up_area'] * 10.7
        else:
            return row['built_up_area']

In [165]:
all_nan_df['built_up_area'] = all_nan_df.apply(convert_scale, axis=1)

In [166]:
all_nan_df.head()

,price,property_type,area,areaWithType,super_built_up_area,built_up_area,carpet_area
9,0.75,house,600.0,Plot area 600(55.74 sq.m.),NaN,600.0,NaN
12,5.60,house,3240.0,Plot area 360(301.01 sq.m.),NaN,3240.0,NaN
15,0.48,house,80.0,Plot area 80(7.43 sq.m.),NaN,80.0,NaN
19,8.50,house,6300.0,Plot area 6300(585.29 sq.m.),NaN,6300.0,NaN
23,0.92,house,603.0,Plot area 67(56.02 sq.m.),NaN,603.0,NaN


In [167]:
# Update original Dataframe

df.update(all_nan_df)

In [168]:
df.isnull().sum()

property_type             0
society                   1
sector                    0
price                    18
price_per_sqft           18
area                     18
areaWithType              0
bedRoom                   0
bathroom                  0
balcony                   0
additionalRoom            0
floorNum                 21
facing                 1127
agePossession             1
nearbyLocations         184
furnishDetails         1001
features                661
super_built_up_area    1935
built_up_area          2079
carpet_area            1891
dtype: int64

In [169]:
df.shape

(3856, 20)

### 2. additionalRoom